# Notebook 4 — Layer Sweep: Which Layer Encodes Truthfulness Best?

## Motivation

So far (Notebooks 2 and 3) we always used the **final layer** (`layer_index = -1`).
That is a reasonable default — the last hidden state has processed the entire context — but it
is not necessarily the best one for probing truth.

The RepE paper shows that **different layers specialise in different abstractions**:

| Layer range | Typical content |
|-------------|----------------|
| Very early (0–2) | Token identity, positional encodings, surface syntax |
| Middle (3–N/2) | Semantic meaning, factual associations, entity linking |
| Late (N/2–N−1) | Task-specific reasoning, probability shaping for next token |

For a truth probe, we expect the **middle-to-late layers** to work best, because that is where
factual knowledge is most accessible.

## What we measure

For each layer $l \in \{0, 1, \ldots, L-1\}$:
1. Extract the hidden vector at the **last token position** of every prompt from layer $l$.
2. Train an LR probe on the training split.
3. Evaluate grouped accuracy on train / validation / test.

Plotting accuracy vs. layer index shows *where* in the network the truthfulness signal peaks.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import io
from contextlib import redirect_stderr, redirect_stdout

from lie_detector_llm.datasets import build_dataset_collection
from lie_detector_llm.experiment import run_layer_sweep
from lie_detector_llm.plotting import plot_layer_sweep

collection = build_dataset_collection(include_repeng_truthful=True)
frame = collection.subset('repeng_truthful')

print(f"Dataset: repeng_truthful")
print(f"Rows   : {len(frame)}")
print(f"Groups : {frame['group_id'].nunique()}")

## Run the layer sweep

We use `distilgpt2` which has **6 transformer layers** (indices 0 – 5).  
Each layer produces a 768-dimensional hidden vector.

In [ ]:
with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
    layer_results = run_layer_sweep(
        frame=frame,
        model_name='distilgpt2',
        probe_method='lr',
    )

# Pivot so rows = layer, columns = split
pivot = layer_results.results.pivot_table(
    index='layer', columns='split', values='grouped_accuracy'
).round(3)
pivot

## Visualise accuracy vs. layer

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

fig, ax = plot_layer_sweep(
    layer_results.results,
    title='LR probe accuracy by layer — repeng_truthful (distilgpt2)',
)
plt.show()

## Best layer on the test set

In [ ]:
test_rows = layer_results.results[layer_results.results['split'] == 'test']
best = test_rows.sort_values('grouped_accuracy', ascending=False).iloc[0]
print(f"Best test layer : {int(best['layer'])}")
print(f"Best test accuracy: {best['grouped_accuracy']:.3f}")
print()
print("Full test layer ranking:")
print(test_rows[['layer', 'grouped_accuracy']].sort_values('grouped_accuracy', ascending=False).to_string(index=False))

## Compare DIM vs LR across layers

DIM (Difference in Means) is unsupervised and may generalise differently than LR.
Let's see if they agree on which layer is best.

In [ ]:
import pandas as pd

all_sweep = []
for method in ['dim', 'lr', 'lat']:
    with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
        sweep = run_layer_sweep(frame=frame, model_name='distilgpt2', probe_method=method)
    all_sweep.append(sweep.results)

combined_sweep = pd.concat(all_sweep, ignore_index=True)

# Show test accuracy for each method × layer
test_sweep = combined_sweep[combined_sweep['split'] == 'test']
pivot_methods = test_sweep.pivot_table(
    index='layer', columns='probe_method', values='grouped_accuracy'
).round(3)
pivot_methods

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(
    data=test_sweep,
    x='layer', y='grouped_accuracy', hue='probe_method',
    style='probe_method', markers=True, dashes=False, ax=ax
)
ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, label='chance')
ax.set_ylim(0, 1.05)
ax.set_title('Test accuracy by layer and probe method (repeng_truthful)')
ax.set_ylabel('Grouped accuracy (test)')
ax.set_xlabel('Layer index')
ax.legend(title='Probe')
plt.tight_layout()
plt.show()

## Interpretation

### What the layer sweep tells us

- **Layer 0** typically has poor accuracy — it sees only the raw token embeddings before any attention.
- **Middle layers** tend to peak — this is where the model has integrated cross-token context
  and associated the candidate answer with factual knowledge.
- **The very last layer** is good but not always the best — it is tuned toward next-token
  probability, not necessarily toward a separable truth geometry.

### Connection to the RepE paper

The RepE paper (Zou et al., 2023) consistently finds that truthfulness probes trained on
middle-to-late layers transfer better across datasets than those trained on early or final
layers. With `distilgpt2` (only 6 layers) the effect is compressed but still visible.

### Practical take-away

When running a probe on a new model, **do not assume the last layer is optimal**.  
A quick layer sweep (as implemented in `run_layer_sweep`) is cheap and can meaningfully
improve both in-distribution and transfer accuracy.